# Mini App Performance Analysis

This notebook parses the SLURM output logs to extract timing and resource metrics.

In [2]:
import os
import re
import pandas as pd
from pathlib import Path

# --- Configuration ---
LOGS_DIR = Path('logs')
MIN_JOB_ID = 536634  # Set your threshold x here

def parse_log_file(file_path):
    try:
        content = file_path.read_text(encoding='utf-8', errors='replace')
    except Exception as e:
        return None
    
    # Extract Job ID from filename
    match = re.search(r'mini_app_output_(\d+)\.txt', file_path.name)
    if not match: return None
    job_id = int(match.group(1))
    
    # Extract path to identify row characteristics
    path_match = re.search(r'Timing and parameters saved to: (.+)/timing_and_parameters\.txt', content)
    if not path_match:
        return None
    
    exp_path = path_match.group(1)
    
    is_cpp_interface = '_cpp_interface_' in exp_path
    force_terrain_upload = 'force_terrain_upload' in exp_path
    
    provider = 'SMARTSIM' # default
    if is_cpp_interface:
        # Extract provider from _cpp_interface_<PROVIDER>_
        provider_match = re.search(r'_cpp_interface_([^_]+)_', exp_path)
        if provider_match:
            provider = provider_match.group(1).upper()
    elif '_SMARTSIM_' in exp_path:
        provider = 'SMARTSIM'
    
    # Extract Timings
    solving_time = None
    solving_match = re.search(r'Solving time: (\d+) seconds', content)
    if solving_match:
        solving_time = int(solving_match.group(1))
    
    # Extract Resource Metrics
    def get_gib(pattern, text):
        m = re.search(pattern, text)
        return float(m.group(1)) if m else None

    ml_input = get_gib(r'ml_input:\s+([0-9.]+) GiB', content)
    ml_output = get_gib(r'ml_output:\s+([0-9.]+) GiB', content)
    ml_preload = get_gib(r'ml_preload:([0-9.]+) GiB', content)
    
    # Network Metrics (rx/tx)
    ib0_rx = None
    ib0_tx = None
    ib0_match = re.search(r'ib0: rx ([0-9.]+) GiB.*?, tx ([0-9.]+) GiB', content)
    if ib0_match:
        ib0_rx = float(ib0_match.group(1))
        ib0_tx = float(ib0_match.group(2))
        
    lo_rx = None
    lo_tx = None
    lo_match = re.search(r'lo: rx ([0-9.]+) GiB.*?, tx ([0-9.]+) GiB', content)
    if lo_match:
        lo_rx = float(lo_match.group(1))
        lo_tx = float(lo_match.group(2))

    return {
        'JobID': job_id,
        'Interface': is_cpp_interface,
        'Provider': provider,
        'ForceUpload': force_terrain_upload,
        'SolvingTime_s': solving_time,
        'ML_Input_GiB': ml_input,
        'ML_Output_GiB': ml_output,
        'ML_Preload_GiB': ml_preload,
        'IB0_RX_GiB': ib0_rx,
        'IB0_TX_GiB': ib0_tx,
        'LO_RX_GiB': lo_rx,
        'LO_TX_GiB': lo_tx
    }

data = []
for f in LOGS_DIR.glob('mini_app_output_*.txt'):
    job_id_match = re.search(r'mini_app_output_(\d+)\.txt', f.name)
    if job_id_match:
        jid = int(job_id_match.group(1))
        if jid >= MIN_JOB_ID:
            row = parse_log_file(f)
            if row:
                data.append(row)

if data:
    df = pd.DataFrame(data)
    df = df.sort_values('JobID').reset_index(drop=True)
else:
    df = pd.DataFrame()

# Display the result
df

,JobID,Interface,Provider,ForceUpload,SolvingTime_s,ML_Input_GiB,ML_Output_GiB,ML_Preload_GiB,IB0_RX_GiB,IB0_TX_GiB,LO_RX_GiB,LO_TX_GiB
0,536634,True,SMARTSIM,False,3278,12.51,0.7,0.00,0.80,16.51,0.0,0.0
1,539973,False,SMARTSIM,True,3173,6.26,0.7,1.25,0.76,9.38,0.0,0.0
2,539976,True,SMARTSIM,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,545217,True,SMARTSIM,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,545242,True,SMARTSIM,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
